# NB01: BacDive → Pangenome Bridge & Coverage Assessment

Link BacDive strains to pangenome species and metal tolerance scores, then compute
coverage waterfall for all phenotype features to determine feasibility.

**Bridge chain**: BacDive taxonomy (species name) → GTDB species name → species_metal_scores.csv

**Reuses**: Species name matching approach from `projects/bacdive_metal_validation/`.

**Go/no-go gate**: If fewer than 500 species have key features after matching, reconsider approach.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

MAIN_REPO = '/home/psdehal/pangenome_science/BERIL-research-observatory'
PROJ = os.path.join(MAIN_REPO, 'projects', 'bacdive_phenotype_metal_tolerance')
BACDIVE = os.path.join(MAIN_REPO, 'data', 'bacdive_ingest')
METAL_ATLAS = os.path.join(MAIN_REPO, 'projects', 'metal_fitness_atlas', 'data')
DATA_OUT = os.path.join(PROJ, 'data')
FIG_OUT = os.path.join(PROJ, 'figures')
os.makedirs(DATA_OUT, exist_ok=True)
os.makedirs(FIG_OUT, exist_ok=True)

## 1. Load BacDive data

In [ ]:
# Core tables
tax = pd.read_csv(os.path.join(BACDIVE, 'taxonomy.tsv'), sep='\t')
strain = pd.read_csv(os.path.join(BACDIVE, 'strain.tsv'), sep='\t')
phys = pd.read_csv(os.path.join(BACDIVE, 'physiology.tsv'), sep='\t')
iso = pd.read_csv(os.path.join(BACDIVE, 'isolation.tsv'), sep='\t')
seq = pd.read_csv(os.path.join(BACDIVE, 'sequence_info.tsv'), sep='\t')
mu = pd.read_csv(os.path.join(BACDIVE, 'metabolite_utilization.tsv'), sep='\t')
enz = pd.read_csv(os.path.join(BACDIVE, 'enzyme.tsv'), sep='\t')

print(f'Strains: {len(strain)}')
print(f'Taxonomy: {len(tax)}, {tax["species"].nunique()} species')
print(f'Physiology: {len(phys)}')
print(f'Isolation: {len(iso)}')
print(f'Sequence info: {len(seq)}')
print(f'Metabolite utilization: {len(mu)}')
print(f'Enzyme: {len(enz)}')

## 2. Load Metal Fitness Atlas scores

In [ ]:
scores = pd.read_csv(os.path.join(METAL_ATLAS, 'species_metal_scores.csv'))
print(f'Metal scores: {len(scores)} species')

# Extract species name from gtdb_species_clade_id
# Format: s__Genus_species--RS_GCF_000742135.1
scores['gtdb_species_name'] = scores['gtdb_species_clade_id'].str.extract(r's__(.+?)--')[0]
scores['gtdb_species_name'] = scores['gtdb_species_name'].str.replace('_', ' ')

# Base name without GTDB suffixes (_A, _B, etc.) for fuzzy matching
scores['gtdb_species_base'] = scores['gtdb_species_name'].str.replace(r' [A-Z]$', '', regex=True)

print(f'Example names: {scores["gtdb_species_name"].head(3).tolist()}')
print(f'Example base:  {scores["gtdb_species_base"].head(3).tolist()}')

## 3. Build species name bridge

Match BacDive species names to GTDB species names (exact match, then base name fallback).
Approach reused from `projects/bacdive_metal_validation/notebooks/01_bacdive_pangenome_bridge.ipynb`.

In [ ]:
# Build lookups: GTDB species name → best metal score (take species with most genomes)
scores_sorted = scores.sort_values('no_genomes', ascending=False)

exact_lookup = scores_sorted.drop_duplicates(subset='gtdb_species_name', keep='first')
exact_lookup = exact_lookup.set_index('gtdb_species_name')[
    ['gtdb_species_clade_id', 'metal_score_raw', 'metal_score_norm', 
     'no_genomes', 'no_gene_clusters', 'n_metal_clusters']
]

base_lookup = scores_sorted.drop_duplicates(subset='gtdb_species_base', keep='first')
base_lookup = base_lookup.set_index('gtdb_species_base')[
    ['gtdb_species_clade_id', 'metal_score_raw', 'metal_score_norm', 
     'no_genomes', 'no_gene_clusters', 'n_metal_clusters']
]

# Match all BacDive strains
records = []
for _, row in tax.iterrows():
    sp = row['species']
    entry = {
        'bacdive_id': row['bacdive_id'],
        'bacdive_species': sp,
        'bacdive_phylum': row['phylum'],
        'bacdive_class': row.get('tax_class', None),
        'bacdive_order': row.get('tax_order', None),
        'bacdive_genus': row['genus'],
    }
    if pd.isna(sp):
        entry['match_type'] = 'no_species'
    elif sp in exact_lookup.index:
        entry['match_type'] = 'exact'
        for col in exact_lookup.columns:
            entry[col] = exact_lookup.loc[sp, col]
    elif sp in base_lookup.index:
        entry['match_type'] = 'base_name'
        for col in base_lookup.columns:
            entry[col] = base_lookup.loc[sp, col]
    else:
        entry['match_type'] = 'no_match'
    records.append(entry)

bridge = pd.DataFrame(records)

# Summary
print('=== Bridge Match Rates ===')
for mt, count in bridge['match_type'].value_counts().items():
    print(f'  {mt:15s}: {count:6d} ({100*count/len(bridge):.1f}%)')

matched = bridge[bridge.match_type.isin(['exact', 'base_name'])]
print(f'\nTotal matched: {len(matched):,} / {len(bridge):,} strains ({100*len(matched)/len(bridge):.1f}%)')
print(f'Unique GTDB species: {matched["gtdb_species_clade_id"].nunique():,}')

## 4. Coverage waterfall

For each phenotype feature, compute how many **species** (not strains) have both a phenotype value AND a metal tolerance score after matching.

In [ ]:
# Get set of matched bacdive_ids and their species
matched_ids = set(matched['bacdive_id'])

def species_coverage(bacdive_ids_with_feature):
    """Given bacdive_ids that have a feature, return count of unique GTDB species with metal scores."""
    overlap = bacdive_ids_with_feature & matched_ids
    return matched[matched['bacdive_id'].isin(overlap)]['gtdb_species_clade_id'].nunique()

# Physiology features (measured only, not predicted)
gram_ids = set(phys[phys['gram_stain'].isin(['positive', 'negative'])]['bacdive_id'])
oxygen_ids = set(phys[phys['oxygen_tolerance'].notna()]['bacdive_id'])
motility_ids = set(phys[phys['motility'].isin(['yes', 'no'])]['bacdive_id'])
shape_ids = set(phys[phys['cell_shape'].notna()]['bacdive_id'])

# Enzyme features (explicit +/- only)
catalase_ids = set(enz[(enz['enzyme_name'].str.contains('catalase', case=False, na=False)) & 
                       (enz['activity'].isin(['+', '-']))]['bacdive_id'])
oxidase_ids = set(enz[(enz['enzyme_name'].str.contains('oxidase', case=False, na=False)) & 
                      ~enz['enzyme_name'].str.contains('cytochrome', case=False, na=False) &
                      (enz['activity'].isin(['+', '-']))]['bacdive_id'])
urease_ids = set(enz[(enz['enzyme_name'].str.contains('urease', case=False, na=False)) & 
                     (enz['activity'].isin(['+', '-']))]['bacdive_id'])

# Metabolite features (explicit +/- only, exclude +/-)
nitrate_ids = set(mu[(mu['compound_name'] == 'nitrate') & 
                     (mu['utilization'].isin(['+', '-']))]['bacdive_id'])
h2s_ids = set(mu[(mu['compound_name'] == 'hydrogen sulfide') & 
                 (mu['utilization'].isin(['+', '-', 'produced']))]['bacdive_id'])
acetate_ids = set(mu[(mu['compound_name'] == 'acetate') & 
                     (mu['utilization'].isin(['+', '-']))]['bacdive_id'])

# Metabolite breadth: strains with at least 1 utilization test
met_breadth_ids = set(mu[mu['utilization'].isin(['+', '-'])]['bacdive_id'])

# Enzyme breadth: strains with at least 1 enzyme test
enz_breadth_ids = set(enz[enz['activity'].isin(['+', '-'])]['bacdive_id'])

# Isolation source
iso_ids = set(iso[iso['cat1'].notna()]['bacdive_id'])

# Build waterfall table
features = [
    ('Gram stain', gram_ids),
    ('Oxygen tolerance', oxygen_ids),
    ('Cell shape', shape_ids),
    ('Motility', motility_ids),
    ('Catalase', catalase_ids),
    ('Oxidase', oxidase_ids),
    ('Urease', urease_ids),
    ('Nitrate reduction', nitrate_ids),
    ('H₂S production', h2s_ids),
    ('Acetate utilization', acetate_ids),
    ('Metabolite breadth', met_breadth_ids),
    ('Enzyme breadth', enz_breadth_ids),
    ('Isolation source', iso_ids),
]

waterfall = []
for name, ids in features:
    n_strains = len(ids)
    n_matched_strains = len(ids & matched_ids)
    n_species = species_coverage(ids)
    waterfall.append({
        'Feature': name,
        'BacDive strains': n_strains,
        'Matched strains': n_matched_strains,
        'Species with metal score': n_species,
    })

wf = pd.DataFrame(waterfall)
print('=== Coverage Waterfall ===')
print(wf.to_string(index=False))
wf.to_csv(os.path.join(DATA_OUT, 'coverage_waterfall.csv'), index=False)

In [ ]:
# Visualization: coverage waterfall
fig, ax = plt.subplots(figsize=(10, 6))

x = range(len(wf))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], wf['Matched strains'], width, 
               label='Matched strains', color='steelblue', alpha=0.7)
bars2 = ax.bar([i + width/2 for i in x], wf['Species with metal score'], width,
               label='Unique species', color='coral', alpha=0.7)

ax.set_xlabel('Phenotype feature')
ax.set_ylabel('Count')
ax.set_title('Coverage Waterfall: BacDive Phenotypes → Metal Tolerance Score')
ax.set_xticks(x)
ax.set_xticklabels(wf['Feature'], rotation=45, ha='right')
ax.legend()

# Add 500-species go/no-go line
ax.axhline(y=500, color='red', linestyle='--', alpha=0.5, label='Go/no-go threshold (500 species)')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, 'coverage_waterfall.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/coverage_waterfall.png')

## 5. Go/no-go assessment

In [ ]:
# Key features for go/no-go
key_features = ['Gram stain', 'Oxygen tolerance', 'Catalase', 'Urease', 'Nitrate reduction']
key_wf = wf[wf['Feature'].isin(key_features)]

min_species = key_wf['Species with metal score'].min()
min_feature = key_wf.loc[key_wf['Species with metal score'].idxmin(), 'Feature']

print('=== GO/NO-GO ASSESSMENT ===')
print(f'Minimum species count among key features: {min_species} ({min_feature})')

if min_species >= 500:
    print('\n✅ GO — Sufficient coverage for all key features.')
    print('Proceeding to phenotype feature engineering (NB02).')
elif min_species >= 200:
    print(f'\n⚠️ CAUTION — {min_feature} has limited coverage ({min_species} species).')
    print('Proceed but interpret results for sparse features with caution.')
else:
    print(f'\n❌ NO-GO — {min_feature} has insufficient coverage ({min_species} species).')
    print('Consider: (1) using GCA accession matching to recover more links,')
    print('         (2) dropping sparse features, or (3) revising the approach.')

In [ ]:
# Save bridge table
bridge.to_csv(os.path.join(DATA_OUT, 'bacdive_pangenome_bridge.csv'), index=False)
print(f'Saved: data/bacdive_pangenome_bridge.csv ({len(bridge):,} rows)')

# Save matched-only subset
matched.to_csv(os.path.join(DATA_OUT, 'matched_strains.csv'), index=False)
print(f'Saved: data/matched_strains.csv ({len(matched):,} rows, {matched["gtdb_species_clade_id"].nunique():,} species)')

print(f'\n=== Summary ===')
print(f'BacDive strains: {len(bridge):,}')
print(f'Matched to pangenome+metal: {len(matched):,} ({100*len(matched)/len(bridge):.1f}%)')
print(f'Unique GTDB species: {matched["gtdb_species_clade_id"].nunique():,}')